# Butane path explorer

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dilawars9/pyscf/blob/dilawar7/geodesic-interpolation/dilawar7/geodesic/notebooks/butane_path_explorer.ipynb)

Generate **101 atom-mapped butane geometries**, inspect a rotatable ball-and-stick
molecule, and track distances, angles and dihedrals. This notebook performs **no
electronic energy or gradient calculations**. A normal CPU runtime is sufficient;
no GPU or Google Drive mount is required.

In Colab, choose **Runtime → Run all**. Installation needs internet access.
Then explore the embedded report or change the parameters and rerun the generation
and display cells. A ZIP containing the report, XYZ paths, measurements and figures
is prepared at the end.

These are idealized trans-to-gauche endpoints, not experimentally measured or
energy-optimized structures. Image index is not physical time. The interpolation
does not certify a minimum-energy path or optimizer convergence.


## Set up the runtime

We install PySCF's prebuilt package, then load only the two new Python modules
from a pinned fork revision. This avoids compiling the full fork. The selected
geodesic backend revision includes its sparse solver. Installation may take
longer than the interpolation itself.

If another PySCF/geodesic version is already imported in this runtime, start a
fresh runtime before running this notebook. The setup reports the modules it
actually loads.


In [ ]:
# @title Install the scientific packages
import os
import subprocess
import sys

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

FORK_URL = 'https://github.com/Dilawars9/pyscf.git'
FORK_COMMIT = 'af7df7057c543e3550bf1506f065126346843c8f'
BACKEND_COMMIT = '476aa041f562d37d5fb4121b86817a52d75e4c5d'
BACKEND_SPEC = (
    'geodesic-interpolate @ git+https://github.com/'
    'virtualzx-nad/geodesic-interpolate.git@' + BACKEND_COMMIT
)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--prefer-binary',
    'pyscf==2.15.0', 'matplotlib>=3.8',
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--force-reinstall', '--no-deps', BACKEND_SPEC,
], check=True)
print('Package installation completed.')


In [ ]:
# @title Load our feature modules from the pinned fork
from pathlib import Path
import tempfile

CHECKOUT = Path(tempfile.gettempdir()) / ('dilawar-pyscf-' + FORK_COMMIT[:12])
if not (CHECKOUT / '.git').is_dir():
    subprocess.run([
        'git', 'clone', '--quiet', '--filter=blob:none', '--no-checkout',
        '--depth', '1', FORK_URL, str(CHECKOUT),
    ], check=True)
    subprocess.run([
        'git', '-C', str(CHECKOUT), 'fetch', '--quiet', '--depth', '1',
        'origin', FORK_COMMIT,
    ], check=True)
    subprocess.run([
        'git', '-C', str(CHECKOUT), 'checkout', '--quiet', '--detach', FORK_COMMIT,
    ], check=True)

actual_commit = subprocess.check_output(
    ['git', '-C', str(CHECKOUT), 'rev-parse', 'HEAD'], text=True,
).strip()
if actual_commit != FORK_COMMIT:
    raise RuntimeError('The cached checkout is at a different revision: ' + str(CHECKOUT))
subprocess.run(['git', '-C', str(CHECKOUT), 'diff', '--quiet', 'HEAD'], check=True)

import pyscf
import pyscf.geomopt
import geodesic_interpolate.geodesic as backend

FEATURE_DIR = CHECKOUT / 'pyscf' / 'geomopt'
for name in ('geodesic', 'path_analysis'):
    full_name = 'pyscf.geomopt.' + name
    expected = FEATURE_DIR / (name + '.py')
    if full_name in sys.modules and Path(sys.modules[full_name].__file__).resolve() != expected.resolve():
        raise RuntimeError('A different feature module is already imported. Restart the runtime.')
if str(FEATURE_DIR) not in pyscf.geomopt.__path__:
    pyscf.geomopt.__path__.insert(0, str(FEATURE_DIR))

from pyscf.geomopt import path_analysis
from pyscf.geomopt.geodesic import interpolate

if not hasattr(backend.Geodesic, 'target_deriv_sparse'):
    raise RuntimeError('The sparse backend is not loaded. Restart the runtime and rerun setup.')

EXAMPLES = CHECKOUT / 'dilawar7' / 'geodesic' / 'examples'
if str(EXAMPLES) not in sys.path:
    sys.path.insert(0, str(EXAMPLES))

print('PySCF wheel:', pyscf.__version__, pyscf.__file__)
print('Feature revision:', actual_commit)
print('Measurement code:', path_analysis.__file__)
print('Backend revision:', BACKEND_COMMIT, '(sparse solver available)')


## Choose the path size

The default is 101 images, including both endpoints. Start with 21 images for a
quick run, or try 201 for denser sampling. Each requested image enters the
geodesic path optimization; extra frames are not inserted just for animation.
Increasing the image count increases work and does not establish convergence.


In [ ]:
# @title Path parameters
NIMAGES = 101 # @param [21, 51, 101, 201] {type:"raw"}
SEED = 7 # @param {type:"integer"}
MAXITER = 50 # @param {type:"integer"}

if not isinstance(NIMAGES, int) or NIMAGES < 3:
    raise ValueError('NIMAGES must be an integer of at least 3.')
if not isinstance(MAXITER, int) or MAXITER < 1:
    raise ValueError('MAXITER must be a positive integer.')
OUTPUT_DIR = Path.cwd() / ('butane-' + str(NIMAGES) + '-images')
print('Results will be written to', OUTPUT_DIR)


In [ ]:
# @title Generate the geometry-only path, measurements, figures and report
import json
import runpy
import numpy as np

previous_argv = sys.argv[:]
try:
    sys.argv = [
        str(EXAMPLES / '03_butane_path.py'),
        '--nimages', str(NIMAGES), '--seed', str(SEED),
        '--maxiter', str(MAXITER), '--output', str(OUTPUT_DIR),
    ]
    runpy.run_path(str(EXAMPLES / '03_butane_path.py'), run_name='__main__')
finally:
    sys.argv = previous_argv

analysis = json.loads((OUTPUT_DIR / 'analysis.json').read_text())
xyz = np.asarray(analysis['coordinates']['geodesic'])
print('Coordinate array:', xyz.shape)
print('Tracked coordinates:', len(analysis['measurements']))
print('Electronic energy calls:', analysis['metadata']['electronic_energy_calls'])
print('Electronic gradient calls:', analysis['metadata']['electronic_gradient_calls'])


## Explore the molecule and synchronized graphs

Drag to rotate the molecule and use its image slider or playback controls.
Click a graph to select an image. Choose a listed coordinate, enter your own atom
numbers, or click **Pick atoms** and select them in order. The viewer uses
**1-based** atom numbers. Carbon atoms are 1–4; hydrogen atoms are 5–14.

The report is isolated in an iframe. If your notebook frontend does not render
the interactive content, the static figure below and downloadable standalone
HTML remain available. Browser downloads may be restricted inside an iframe;
the final notebook cell provides a ZIP download independently of the viewer.


In [ ]:
from IPython.display import HTML, Image, display
import html

report_html = (OUTPUT_DIR / 'report.html').read_text()
iframe_html = (
    '<iframe title="Butane molecular path explorer" '
    'sandbox="allow-scripts allow-downloads" '
    'style="width:100%;height:1250px;border:1px solid #dce4ee;border-radius:8px" '
    'srcdoc="' + html.escape(report_html, quote=True) + '"></iframe>'
)
display(HTML(iframe_html))


## Where the bond-length, angle and dihedral code lives

The reusable calculations are in
[`pyscf/geomopt/path_analysis.py`](https://github.com/Dilawars9/pyscf/blob/dilawar7/geodesic-interpolation/pyscf/geomopt/path_analysis.py).
The Python API uses **0-based** atom indices and accepts an array of shape
`(number_of_images, number_of_atoms, 3)`. Distances inherit the input coordinate
unit (Angstrom here); angles and signed torsions use degrees.

The next cell converts the viewer's **1-based** atom numbers to Python indices
and calls the real functions. Edit the selections and rerun it to track another
coordinate. An undefined angle or torsion is `NaN`; it is not assigned zero.
Unwrapping assumes changes smaller than 180 degrees between neighboring images
and preserves gaps where the torsion is undefined.


In [ ]:
# @title Extract selected geometric coordinates directly in Python
DISTANCE_ATOMS = [1, 4] # @param {type:"raw"}
ANGLE_ATOMS = [1, 2, 3] # @param {type:"raw"}
DIHEDRAL_ATOMS = [1, 2, 3, 4] # @param {type:"raw"}

def atom_selection(numbers):
    if any(not isinstance(i, int) or i < 1 or i > xyz.shape[1] for i in numbers):
        raise ValueError('Atom numbers must be integers between 1 and ' + str(xyz.shape[1]))
    return [tuple(i - 1 for i in numbers)]

bond_length = path_analysis.distances(xyz, atom_selection(DISTANCE_ATOMS))[:, 0]
bond_angle = path_analysis.angles(xyz, atom_selection(ANGLE_ATOMS))[:, 0]
wrapped_torsion = path_analysis.dihedrals(xyz, atom_selection(DIHEDRAL_ATOMS))
dihedral = path_analysis.unwrap_dihedrals(wrapped_torsion)[:, 0]
rmsd = path_analysis.aligned_rmsd(xyz)

print('image   distance / A   angle / deg   unwrapped dihedral / deg   aligned RMSD / A')
for i in sorted({0, len(xyz) // 2, len(xyz) - 1}):
    print(f'{i:5d}   {bond_length[i]:12.6f}   {bond_angle[i]:11.6f}   '
          f'{dihedral[i]:24.6f}   {rmsd[i]:16.6f}')

import matplotlib.pyplot as plt
figure, axes = plt.subplots(1, 3, figsize=(13, 3.5), layout='constrained')
for axis, values, label in zip(
    axes, (bond_length, bond_angle, dihedral),
    ('Distance (Angstrom)', 'Angle (degrees)', 'Unwrapped dihedral (degrees)'),
):
    axis.plot(values, color='#087f8c', linewidth=2)
    axis.set(xlabel='Image index', ylabel=label)
    axis.grid(alpha=.2)
from io import BytesIO
plot_buffer = BytesIO()
figure.savefig(plot_buffer, format='png', dpi=140)
display(Image(data=plot_buffer.getvalue()))
plt.close(figure)


In [ ]:
# Static views of the path and the Cartesian comparison are available as well.
display(Image(filename=str(OUTPUT_DIR / 'geometry_summary.png')))


## Check the generated path

These checks validate shapes, unchanged endpoint internal geometries and the
known 180° → 60° backbone torsion. They do not establish energetic accuracy or
optimizer convergence. The fixed bond graph is supplied explicitly; no bond
orders or chemical reactions are inferred from distances.


In [ ]:
from butane_geometry import endpoints

symbols, initial, final, reference_bonds = endpoints()
assert xyz.shape == (NIMAGES, 14, 3)
assert np.isfinite(xyz).all()
for actual, expected in ((xyz[0], initial), (xyz[-1], final)):
    actual_distances = np.linalg.norm(actual[:, None] - actual, axis=-1)
    expected_distances = np.linalg.norm(expected[:, None] - expected, axis=-1)
    np.testing.assert_allclose(actual_distances, expected_distances, atol=2e-9)
backbone_torsion = path_analysis.dihedrals(xyz, [(0, 1, 2, 3)])[:, 0]
np.testing.assert_allclose(backbone_torsion[[0, -1]], [180, 60], atol=1e-7)
assert analysis['metadata']['electronic_energy_calls'] == 0
assert analysis['metadata']['electronic_gradient_calls'] == 0
print('Path checks passed. No electronic energy or gradient calculations were requested.')
print(json.dumps(analysis['metadata'], indent=2))


## Save the results

The next cell creates a ZIP containing the full HTML report, XYZ paths, coordinate
tables, JSON metadata and PNG/SVG figures. In Colab, set **DOWNLOAD_RESULTS** to
`True` to download it. You can also use Colab's Files pane. Download the HTML
before opening it outside the notebook; GitHub displays HTML source.

This bundle contains the generated predefined measurements. Custom coordinates
added interactively inside the iframe can be exported using the viewer's CSV
button or copied from its export-text fallback.


In [ ]:
# @title Bundle and optionally download the results
DOWNLOAD_RESULTS = False # @param {type:"boolean"}
import shutil
from IPython.display import FileLink

archive = Path(shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR))
print('Results ZIP:', archive.name, f'({archive.stat().st_size / 1024:.0f} KiB)')
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
    except ImportError:
        display(FileLink(archive.name))
    else:
        files.download(str(archive))
else:
    print('Set DOWNLOAD_RESULTS = True and rerun this cell, or use the file browser.')


## Source and scientific scope

- [PySCF fork and worked examples](https://github.com/Dilawars9/pyscf/tree/dilawar7/geodesic-interpolation/dilawar7/geodesic)
- [Geodesic interpolation backend](https://github.com/virtualzx-nad/geodesic-interpolate)
- [3Dmol.js](https://3dmol.org/), bundled with its license notices inside the report.
- [PySCF installation documentation](https://pyscf.org/user/install.html)

Endpoint construction uses C–C = 1.54 Angstrom, C–H = 1.09 Angstrom and ideal
tetrahedral angles. They are model conformers, not optimized minima. The pinned
feature revision and backend revision are displayed above; notebook setup uses
PySCF's wheel for native libraries, not a clean native build of the fork.

Developed with OpenAI Codex assistance for implementation, documentation and tests.
Local notebook execution and hosted Colab execution are distinct validation steps;
consult the repository's validation record for what has actually been checked.
